# Chapter 4: join four sources on batch ID

The chapter asks you to map one plant on one page. This notebook does the same on a synthetic plant extract: the ERP brew sheet, the historian, the lab and the lots and sales. Synthetic data only; no real brewery's numbers.

What to look at: how many batches fail to join and why.

In [ ]:
import pandas as pd

brews = pd.read_csv('../data/plant_brews.csv')
historian = pd.read_csv('../data/plant_historian.csv')
lab = pd.read_csv('../data/plant_lab.csv')
lots = pd.read_csv('../data/plant_lots.csv')
sales = pd.read_csv('../data/plant_sales.csv')
for name, df in [('brews', brews), ('historian', historian), ('lab', lab), ('lots', lots), ('sales', sales)]:
    print(name, df.shape)

## Join brews to the historian and the lab

A left join keeps every brew. A missing value on the right means the batch did not match.

In [ ]:
j = (brews.merge(historian, on='batch_id', how='left', indicator='in_historian')
         .merge(lab, on='batch_id', how='left', indicator='in_lab'))
print('No historian row:', (j.in_historian == 'left_only').sum())
print('No lab row:', (j.in_lab == 'left_only').sum())
print('Lab IDs that match no brew:', sorted(set(lab.batch_id) - set(brews.batch_id)))

The lab typed some IDs without the dash or in lower case. Fix the key, not the analysis: one rule, written down.

In [ ]:
def clean_id(s):
    s = s.upper().replace('-', '')
    return s[:3] + '-' + s[3:]

lab['batch_id'] = lab.batch_id.map(clean_id)
j = brews.merge(lab, on='batch_id', how='left', indicator=True)
print('No lab row after cleaning:', (j._merge == 'left_only').sum())

## Sales know a lot code, not a batch

Sales join to lots, lots join to brews. Count the invoice lines that reach no batch.

In [ ]:
s = sales.merge(lots, on='lot_code', how='left')
print('Invoice lines with no batch:', s.batch_id.isna().sum())
print(s[s.batch_id.isna()].lot_code.unique())

## Units: one base unit

Some rows were typed in barrels. Convert at the edge with one factor (1.1734 hl per bbl).

In [ ]:
brews['knockout_hl'] = brews.knockout_volume.where(brews.volume_unit == 'hl', (brews.knockout_volume * 1.1734).round(1))
print(brews[brews.volume_unit == 'bbl'][['batch_id', 'knockout_volume', 'knockout_hl']])

## Whose number wins

Keep the lab and the line side by side with a gap column. Batch B24-118 is the 4.2 example.

In [ ]:
gap = historian.merge(lab, on='batch_id')
gap['gap_plato'] = (gap.og_plato_lab - gap.line_og_plato).round(2)
print(gap[gap.batch_id == 'B24-118'][['batch_id', 'og_plato_lab', 'line_og_plato', 'gap_plato']])
print('Typical gap:', gap.gap_plato.median())